
# Claude API Tool Use and Streaming
## A simple guide based on the provided notebooks

This notebook combines the helper functions, tool definitions, conversation loop, tool execution, error handling, `tool_choice`, and streaming ideas from:

- `003_tool_streaming.ipynb`
- `003_tool_streaming_completed.ipynb`
- the three helper-function screenshots

We will use one example throughout:

> **Ask Claude to create a fake computer science article and save it with a Python function.**

### The whole process in one line

```text
User → Claude → tool_use request → Python runs the tool
     → tool_result → Claude gives the final response
```

**Important:** Claude does not directly run your Python function. Claude only asks for a tool to be run. Your code decides whether and how to run it.



## 1. Install and configure the SDK

Install the required packages once:

```python
%pip install anthropic python-dotenv
```

Create a file named `.env` beside this notebook:

```text
ANTHROPIC_API_KEY=your_api_key_here
ANTHROPIC_MODEL=claude-sonnet-4-5
```

Keeping the key in `.env` is safer than writing it directly in the notebook.


In [ ]:

# Load the API key and create the Anthropic client.
import copy
import json
import os

from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()

# This uses the model from your original notebooks by default.
# You can replace it through the ANTHROPIC_MODEL environment variable.
MODEL = os.getenv("ANTHROPIC_MODEL", "claude-sonnet-4-5")

print("Client created.")
print("Model:", MODEL)



## 2. Store the conversation in a `messages` list

The Messages API receives the conversation as a list.

Each message has:

- a `role`: either `"user"` or `"assistant"`
- `content`: a list of content blocks

Content blocks can contain normal text, a tool request, or a tool result.


In [ ]:

def add_user_message(messages, message):
    """Add normal text or tool-result blocks as a user message."""
    if isinstance(message, list):
        content = message
    else:
        content = [{"type": "text", "text": message}]

    messages.append({
        "role": "user",
        "content": content,
    })


def add_assistant_message(messages, message):
    """Convert an SDK response into reusable conversation history."""
    if isinstance(message, list):
        content = message

    elif hasattr(message, "content"):
        content = []

        for block in message.content:
            if block.type == "text":
                content.append({
                    "type": "text",
                    "text": block.text,
                })

            elif block.type == "tool_use":
                content.append({
                    "type": "tool_use",
                    "id": block.id,
                    "name": block.name,
                    "input": block.input,
                })

    else:
        content = [{"type": "text", "text": str(message)}]

    messages.append({
        "role": "assistant",
        "content": content,
    })


def text_from_message(message):
    """Return only the normal text blocks from an SDK message."""
    return "\n".join(
        block.text
        for block in message.content
        if block.type == "text"
    )


In [ ]:

# Small example that does not call the API.
messages = []

add_user_message(
    messages,
    "Create and save a fake computer science article.",
)

messages



## 3. A basic non-streaming `chat()` helper

This helper gathers the common API settings in one place.

The main arguments from the screenshots are:

- `messages`: conversation history
- `system`: optional high-level instructions
- `temperature`: controls variation
- `stop_sequences`: text that can stop generation
- `tools`: tools Claude is allowed to request
- `tool_choice`: whether a tool is optional or forced


In [ ]:

def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=None,
    tools=None,
    tool_choice=None,
):
    """Send one normal, non-streaming Messages API request."""
    params = {
        "model": MODEL,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    if tools:
        params["tools"] = tools

    if tool_choice:
        params["tool_choice"] = tool_choice

    if system:
        params["system"] = system

    return client.messages.create(**params)



## 4. Define the tool Claude is allowed to request

A client-side tool definition contains three important parts:

1. `name`: the name Claude will request
2. `description`: what the tool does
3. `input_schema`: the JSON-shaped inputs the tool expects

The schema below requires:

- a short `abstract`
- a nested `meta` object
- an integer `word_count`
- a text `review`

Good descriptions matter because they tell Claude what kind of values to generate.


In [ ]:

save_article_schema = {
    "name": "save_article",
    "description": "Save a scholarly journal article.",
    "input_schema": {
        "type": "object",
        "properties": {
            "abstract": {
                "type": "string",
                "description": "The article abstract. One short sentence maximum.",
            },
            "meta": {
                "type": "object",
                "properties": {
                    "word_count": {
                        "type": "integer",
                        "description": "The article's word count.",
                    },
                    "review": {
                        "type": "string",
                        "description": "An eight-sentence review of the paper.",
                    },
                },
                "required": ["word_count", "review"],
            },
        },
        "required": ["abstract", "meta"],
    },
}


# The original notebook also contained a shorter version.
# Only the review description changes.
save_short_article_schema = copy.deepcopy(save_article_schema)
save_short_article_schema["input_schema"]["properties"]["meta"][
    "properties"
]["review"]["description"] = "A one-sentence review of the paper."



## 5. Write the real Python function

The schema only describes the tool to Claude. We still need a real function.

This example stores articles in a Python list so it is safe and easy to inspect.

The function also validates the most important values before saving them. This is useful because model-generated tool inputs should not be trusted automatically.


In [ ]:

saved_articles = []


def save_article(abstract, meta):
    """Validate and save an article in memory."""
    if not isinstance(abstract, str) or not abstract.strip():
        raise ValueError("abstract must be a non-empty string")

    if not isinstance(meta, dict):
        raise ValueError("meta must be an object/dictionary")

    if not isinstance(meta.get("word_count"), int):
        raise ValueError("meta.word_count must be an integer")

    if not isinstance(meta.get("review"), str):
        raise ValueError("meta.review must be a string")

    article = {
        "abstract": abstract,
        "meta": meta,
    }

    saved_articles.append(article)

    return {
        "status": "saved",
        "article_number": len(saved_articles),
    }



## 6. Match tool requests to Python functions

Claude may return one or more `tool_use` blocks.

For each request, the code must:

1. read the tool name and input
2. run the matching Python function
3. create a `tool_result` block
4. use the original tool request's `id` as `tool_use_id`

`is_error=True` lets Claude know the tool failed.


In [ ]:

TOOL_FUNCTIONS = {
    "save_article": save_article,
}


def run_tool(tool_name, tool_input):
    """Run one registered Python tool."""
    function = TOOL_FUNCTIONS.get(tool_name)

    if function is None:
        raise ValueError(f"Unknown tool: {tool_name}")

    return function(**tool_input)


def run_tools(message):
    """Run every tool_use block found in an assistant response."""
    tool_requests = [
        block
        for block in message.content
        if block.type == "tool_use"
    ]

    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(
                tool_request.name,
                tool_request.input,
            )

            result = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False,
            }

        except Exception as error:
            result = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {error}",
                "is_error": True,
            }

        tool_result_blocks.append(result)

    return tool_result_blocks



## 7. Build the complete non-streaming conversation loop

One API request may not finish the task.

When Claude requests a tool, the application must send the result back and call Claude again. The loop ends when `stop_reason` is no longer `"tool_use"`.

The assistant's tool request must be added to history **before** its matching user `tool_result`.


In [ ]:

def run_conversation(
    messages,
    tools=None,
    tool_choice=None,
    max_turns=10,
):
    """Run Claude, execute requested tools, and continue to a final answer."""
    current_tool_choice = tool_choice

    for _ in range(max_turns):
        response = chat(
            messages=messages,
            tools=tools,
            tool_choice=current_tool_choice,
        )

        # Keep Claude's text/tool request in the history.
        add_assistant_message(messages, response)

        response_text = text_from_message(response)
        if response_text:
            print(response_text)

        # No tool request means the conversation turn is complete.
        if response.stop_reason != "tool_use":
            return messages

        # Run all requested tools and return their results as a user message.
        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

        # Force a tool only on the first request.
        # Otherwise, forcing it every turn could create repeated tool calls.
        current_tool_choice = None

    raise RuntimeError("Maximum tool-use turns reached.")



## 8. Stream Claude's response

Streaming shows output while Claude is generating it instead of waiting for the whole response.

The original streaming notebook handled these useful events:

- `text`: a piece of normal text
- `content_block_start`: a text or tool block begins
- `input_json`: part of the tool's JSON input
- `content_block_stop`: the block ends
- `stream.get_final_message()`: the completed SDK message

The JSON fragments printed during streaming are only pieces. Do not treat each piece as a complete JSON object.


In [ ]:

def chat_stream(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=None,
    tools=None,
    tool_choice=None,
    fine_grained=False,
):
    """Create a streaming Messages API request."""
    stream_tools = copy.deepcopy(tools) if tools else None

    # Current tool-level form of fine-grained input streaming.
    # The original notebook used the legacy beta header instead.
    if fine_grained and stream_tools:
        for tool in stream_tools:
            tool["eager_input_streaming"] = True

    params = {
        "model": MODEL,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    if stream_tools:
        params["tools"] = stream_tools

    if tool_choice:
        params["tool_choice"] = tool_choice

    if system:
        params["system"] = system

    return client.messages.stream(**params)



### Fine-grained tool streaming

Fine-grained streaming sends tool input fragments as Claude creates them. It can reduce the wait before a large tool argument begins appearing.

Your original notebook enabled it with:

```python
client.beta.messages.stream(
    ...,
    betas=["fine-grained-tool-streaming-2025-05-14"],
)
```

The current API also supports enabling it on an individual tool with:

```python
tool["eager_input_streaming"] = True
```

Fine-grained fragments can be incomplete or invalid JSON. Wait for the completed message, validate the final input, and return an error tool result instead of running malformed input.


In [ ]:

def run_streaming_conversation(
    messages,
    tools=None,
    tool_choice=None,
    fine_grained=False,
    max_turns=10,
):
    """Streaming version of the full tool-use loop."""
    current_tool_choice = tool_choice

    for _ in range(max_turns):
        with chat_stream(
            messages=messages,
            tools=tools,
            tool_choice=current_tool_choice,
            fine_grained=fine_grained,
        ) as stream:

            for event in stream:
                if event.type == "text":
                    print(event.text, end="", flush=True)

                elif event.type == "content_block_start":
                    if event.content_block.type == "tool_use":
                        print(
                            f'\n>>> Tool call: "{event.content_block.name}"'
                        )

                elif event.type == "input_json":
                    if event.partial_json:
                        print(event.partial_json, end="", flush=True)

                elif event.type == "content_block_stop":
                    print()

            response = stream.get_final_message()

        add_assistant_message(messages, response)

        if response.stop_reason != "tool_use":
            return messages

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

        # A forced tool choice is used for the first model call only.
        current_tool_choice = None

    raise RuntimeError("Maximum tool-use turns reached.")



## 9. Run the complete example

The following cell:

1. creates an empty conversation
2. adds the user's request
3. lets Claude decide whether to call `save_article`
4. runs the function if requested
5. sends the result back to Claude
6. prints the streamed response

Running this cell sends requests to the Claude API and may use API credits.


In [ ]:

messages = []

add_user_message(
    messages,
    "Create and save a fake computer science article.",
)

final_messages = run_streaming_conversation(
    messages=messages,
    tools=[save_article_schema],
)

print("\nSaved articles:", len(saved_articles))



## 10. Control tool usage with `tool_choice`

Common choices are:

| Setting | Meaning |
|---|---|
| omitted or `{"type": "auto"}` | Claude decides whether to use a tool |
| `{"type": "any"}` | Claude must use one of the provided tools |
| `{"type": "tool", "name": "save_article"}` | Claude must request that exact tool |
| `{"type": "none"}` | Claude cannot request a tool |

The completed notebook forced `save_article`:

```python
tool_choice={"type": "tool", "name": "save_article"}
```

Forcing a tool is useful when structured tool input is required. In a normal multi-turn loop, force it only on the first request and then allow Claude to respond normally after receiving the tool result.


In [ ]:

forced_messages = []

add_user_message(
    forced_messages,
    "Create and save a fake computer science article.",
)

run_streaming_conversation(
    messages=forced_messages,
    tools=[save_article_schema],
    tool_choice={"type": "tool", "name": "save_article"},
)



## 11. Why the completed notebook showed malformed JSON

The completed notebook intentionally asked Claude to include:

```text
"word_count": undefined
```

This demonstrates a bad JavaScript-style value. JSON does **not** have an `undefined` value.

It also produced `meta` as a string instead of the object required by the schema. The original `save_article(**kwargs)` function accepted anything and returned `"Article saved!"`, so the malformed structure was not rejected.

The safer `save_article()` in this notebook validates the input. `run_tools()` catches validation failures and returns a `tool_result` with `is_error=True`.


In [ ]:

# Local demonstration: this fails before any API call is made.
malformed_json = '{"word_count": undefined}'

try:
    json.loads(malformed_json)
except json.JSONDecodeError as error:
    print("Invalid JSON:", error)



## 12. What each helper function does

| Helper | Purpose |
|---|---|
| `add_user_message()` | Adds user text or tool results to history |
| `add_assistant_message()` | Saves Claude's text and tool requests |
| `text_from_message()` | Extracts normal text from an SDK response |
| `chat()` | Makes one non-streaming API request |
| `chat_stream()` | Makes one streaming API request |
| `run_tool()` | Matches a tool name to a Python function |
| `run_tools()` | Executes all tool requests and builds results |
| `run_conversation()` | Repeats non-streaming calls until finished |
| `run_streaming_conversation()` | Repeats streaming calls until finished |

## 13. Main takeaways

1. A tool definition describes a function; it does not execute it.
2. Claude returns a `tool_use` content block when it wants a tool.
3. Your Python code runs the function.
4. The result is returned in a user `tool_result` block.
5. The `tool_use_id` must match the original tool request ID.
6. Continue calling Claude until `stop_reason` is not `"tool_use"`.
7. Streaming changes when output appears, not the basic tool-use loop.
8. Fine-grained tool input may be incomplete or invalid, so validate it.
9. Use `is_error=True` when a tool cannot be run.
10. Never execute model-generated tool input without checking it first.



## Reference notes

This guide keeps the structure and `save_article` example from the supplied notebooks while making the conversation loop safer and easier to follow.

Official Claude documentation used to verify the current streaming form:

- [How tool use works](https://platform.claude.com/docs/en/agents-and-tools/tool-use/overview)
- [Handle tool calls](https://platform.claude.com/docs/en/agents-and-tools/tool-use/handle-tool-calls)
- [Streaming Messages](https://platform.claude.com/docs/en/build-with-claude/streaming)
- [Fine-grained tool streaming](https://platform.claude.com/docs/en/agents-and-tools/tool-use/fine-grained-tool-streaming)
